# Local Outlier Factor - Dataset de Churn

In [1]:
import pandas as pd
import numpy as np

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import recall_score

## Ler Dados

In [2]:
df_churn = pd.read_csv("./dataset/churn.csv")
df_churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   IDCliente         7032 non-null   str    
 1   Genero            7032 non-null   str    
 2   Mais65anos        7032 non-null   int64  
 3   TemParceiro       7032 non-null   str    
 4   TemDependentes    7032 non-null   str    
 5   PhoneService      7032 non-null   str    
 6   MultipleLines     7032 non-null   str    
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   str    
 9   OnlineBackup      7032 non-null   str    
 10  DeviceProtection  7032 non-null   str    
 11  TechSupport       7032 non-null   str    
 12  StreamingTV       7032 non-null   str    
 13  StreamingMovies   7032 non-null   str    
 14  tenure            7032 non-null   int64  
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  Paymen

In [3]:
df_churn.head(n=10)

,IDCliente,Genero,Mais65anos,TemParceiro,TemDependentes,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,No,No phone service,DSL,No,Yes,...,No,No,No,1,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,Yes,No,DSL,Yes,No,...,No,No,No,34,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,Yes,No,DSL,Yes,Yes,...,No,No,No,2,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,No,No phone service,DSL,Yes,No,...,Yes,No,No,45,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,Yes,No,Fiber optic,No,No,...,No,No,No,2,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,Yes,Yes,Fiber optic,No,No,...,No,Yes,Yes,8,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
6,1452-KIOVK,Male,0,No,Yes,Yes,Yes,Fiber optic,No,Yes,...,No,Yes,No,22,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No
7,6713-OKOMC,Female,0,No,No,No,No phone service,DSL,Yes,No,...,No,No,No,10,Month-to-month,No,Mailed check,29.75,301.90,No
8,7892-POOKP,Female,0,Yes,No,Yes,Yes,Fiber optic,No,No,...,Yes,Yes,Yes,28,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,Yes,No,DSL,Yes,Yes,...,No,No,No,62,One year,No,Bank transfer (automatic),56.15,3487.95,No


## Contar quantos clientes que fizeram churn

In [4]:
df_churn.value_counts('Churn')

Churn
No     5163
Yes    1869
Name: count, dtype: int64

In [5]:
df_churn.value_counts('Churn', normalize=True) * 100

Churn
No     73.421502
Yes    26.578498
Name: proportion, dtype: float64

## Preparação dos Dados

In [6]:
X = df_churn.drop(columns=['IDCliente', 'Churn'])
y = df_churn['Churn']

# Função de Transformação
def binary_transformer_function(X):
    return X.map(lambda x: 1 if x == 'Yes' else 0)

numeric_features = ['tenure', 'TotalCharges', 'MonthlyCharges']
categorical_features = [
    'Genero',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'StreamingMovies',
    'StreamingTV',
    'Contract',
    'PaymentMethod',
]
binary_features = [
    'TemParceiro',
    'TemDependentes',
    'TechSupport',
    'PhoneService',
    'PaperlessBilling',
]
no_transformation_features = ['Mais65anos']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
binary_transformer = FunctionTransformer(binary_transformer_function)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('bin', binary_transformer, binary_features),
    ('pass', 'passthrough', no_transformation_features),
])

X_transformed = preprocessor.fit_transform(X=X)

X_transformed.shape

(7032, 39)

## Treinar algoritmo Local Outlier Factor

In [7]:
lof = LocalOutlierFactor(
  n_neighbors=20,
  contamination=0.26, # usa esse numero pois é cerca dessa porcentagem dos dados que é churn
)

In [8]:
y_pred = lof.fit_predict(X=X_transformed)

### Mostrar valores preditos
- -1: Pontos anômalos.
- +1: Pontos normais.
- `negative_outlier_factor_` é o inverso do LOF. Quanto menor, mais anormal.

In [9]:
y_pred[:17]

array([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1, -1])

#### Mostrar o negative outlier factor calculado para cada ponto de dados

In [10]:
lof.negative_outlier_factor_[:17]

array([-1.0238333 , -1.03547225, -1.02610568, -1.00226765, -1.00020554,
       -1.00794044, -1.09738768, -1.02264304, -1.00386196, -0.99510011,
       -1.0253553 , -1.07628362, -1.02206732, -1.01123275, -1.11696329,
       -1.00900062, -1.16090685])

## Apresentar resultados

### Identificar Anomalias

In [11]:
outliers = y_pred == -1 # pontos normais
inliers = y_pred == 1 # pontos anormais

num_outliers = np.sum(outliers)
num_inliers = np.sum(inliers)

print(f"Anomalias detectadas: {num_outliers}")
print(f"Pontos normais detectados: {num_inliers}")

Anomalias detectadas: 1829
Pontos normais detectados: 5203


### Recall-Score

#### Converter y para mesma base de y_pred

In [12]:
y_true = y.map(lambda x: -1 if x == 'Yes' else 1)

### Calcular Score com vase no valor de y (Churn real da base)
- Usar Recall, pois o objetivo original era maximizar True Positive Rate

In [13]:
recall_score(y_pred=y_pred, y_true=y_true)

0.7515010652721286